# Nuclear segmentation in the UVA DeepCell environment

Author and pipeline lead: Yang Cheng, University of Virginia (UVA), Christopher Deppmann and Melanie Worley laboratories.

Developed with assistance from ChatGPT (OpenAI) in coding, documentation and tutorial preparation.

This notebook reconstructs the accepted workflow recovered from the original UVA HPC notebooks. It has not been executed locally. The original run used Python 3.10.20, DeepCell 0.12.10, TensorFlow 2.8.4 and an NVIDIA RTX 3090. Select the original Jupyter kernel **DeepCell ICC GPU** (`deepcell_icc_gpu`). Original scripts and source-only notebooks are included in `source_workflows/`; see that folder's README before running historical trial cells. This is a record of the working environment, not a complete installation lock.

Keep this notebook beside `code/`. Use your own institution's authentication and DeepCell model access. If model retrieval requests authentication, configure it in your environment according to the service instructions. Never paste a key into this notebook, a shared script, an output or GitHub. A cached model can behave differently from a new download; model access is not tested here.

The full projected images are not included in the downloadable tutorial. Supply your own four-channel CYX TIFF projections from FIJI. The small bundled crop is for measuring existing masks, not for reproducing full-plane segmentation. Start with one representative plane and review the outputs before a batch run.

## Record your DeepCell environment

The original kernel was DeepCell ICC GPU (`deepcell_icc_gpu`). The recovered environment record lists observed versions; it does not contain a complete Conda export, CUDA runtime configuration or scheduler allocation. In that kernel, run the following cell to capture the installed package versions and TensorFlow GPU visibility without model inference. Preserve the output with your run. This helper does not read API keys or other environment variables.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'code'))
from capture_deepcell_environment import capture_environment
import json
record = capture_environment()
with Path('deepcell_environment_capture.json').open('x') as handle:
    json.dump(record, handle, indent=2, default=str)
print('Saved the active environment record; no model loaded.')

In [ ]:
from pathlib import Path
import sys, json, copy
import numpy as np
import tifffile
import matplotlib.pyplot as plt

TUTORIAL = Path.cwd()  # folder containing this notebook and code/
sys.path.insert(0, str(TUTORIAL / 'code'))
from nuclear_segmentation import (
    ENHANCEMENT, SETTINGS, enhance_dapi, load_nuclear_model,
    predict_nuclei, filter_nuclei, save_segmentation,
)

INPUT = Path('/replace/with/your/projected_images')
OUTPUT = Path('/replace/with/a/new/segmentation_run')
files = {
    'top': INPUT / 'top_Z28-Z29_4channels.tif',
    'middle': INPUT / 'middle_Z33-Z34_4channels.tif',
    'bottom': INPUT / 'bottom_Z38-Z39_4channels.tif',
}
DAPI_CHANNEL = 2  # zero-based: C3 in this dataset
cfg = copy.deepcopy(SETTINGS)
cfg['pixel_size_um'] = 0.5546006944444444  # replace for your acquisition
print(json.dumps(cfg, indent=2))

## Load raw DAPI and enhance nuclear cores

Do not alter raw intensity channels. Enhancement makes a separate segmentation input: Gaussian smoothing (sigma 0.5 px), subtracting a broad background (sigma 12 px), clipping at zero, then a nonnegative scale-normalized negative Laplacian of Gaussian (sigma 1.8 px). These are example settings for this acquisition and should be reviewed on new tissues.

In [ ]:
plane = 'middle'
with tifffile.TiffFile(files[plane]) as tf:
    assert tf.series[0].axes == 'CYX', 'Export a projected CYX TIFF first'
    raw = tf.asarray()
assert raw.ndim == 3 and 0 <= DAPI_CHANNEL < raw.shape[0]
dapi = raw[DAPI_CHANNEL]
core = enhance_dapi(dapi, **ENHANCEMENT)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, image, title in zip(axes, [dapi, core], ['Raw DAPI', 'Enhanced core response']):
    ax.imshow(image, cmap='gray', vmin=0, vmax=np.percentile(image, 99.5))
    ax.set_title(title); ax.axis('off')
plt.show()

## Load the pretrained model and predict

The original workflow used `NuclearSegmentation.from_version('1.1')`. It did not train a new model. The core image is enlarged 3× with linear interpolation, predicted with `image_mpp=app.model_mpp`, and mapped back by nearest-neighbor interpolation. This preserves an empirically selected apparent nuclear size. It is not a conversion of the microscope calibration. Output measurements use the original physical pixel size.

For new acquisitions, review this setting along with the enhancement and watershed parameters. The original settings do not establish general segmentation accuracy. This cell runs GPU inference in the selected HPC environment.

In [ ]:
app = load_nuclear_model()
initial_labels = predict_nuclei(core, app, cfg)
print('Detected labels before tissue filtering:', np.count_nonzero(np.unique(initial_labels)))

## Apply tissue and boundary filters, then review

A smoothed raw DAPI image is thresholded at 0.5× Otsu, closed with a disk of radius 32 px, stripped of components smaller than 5,000 px and hole-filled. Nuclei need at least 50% overlap with this tissue mask.

For nuclei intersecting a 10 px tissue-boundary band, compare median raw nuclear DAPI with median local background in a 2–5 px ring. Other detected nuclei are excluded from that ring. With at least 10 background pixels, a ratio below 1.2 flags a nucleus for exclusion. Unevaluable nuclei are retained. These filters can remove true weak nuclei, so inspect flags, splits, merges and anatomy on your own images. The original exclusions were visually reviewed; the helper applies the recorded rule automatically and returns its QC records for review.

In [ ]:
labels, tissue, summary, boundary_qc = filter_nuclei(dapi, initial_labels, cfg)
print(json.dumps(summary, indent=2))
from skimage.segmentation import find_boundaries
flagged_ids = [r['label'] for r in boundary_qc if r['flagged']]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
limits = dict(cmap='gray', vmin=0, vmax=np.percentile(dapi, 99.5))
for ax in axes: ax.imshow(dapi, **limits); ax.axis('off')
axes[0].contour(tissue, levels=[0.5], colors='yellow', linewidths=0.4)
axes[0].set_title('Tissue boundary')
for ax, mask, title, color in [
    (axes[1], initial_labels, 'Initial detections', 'cyan'),
    (axes[2], labels, 'Retained nuclei; magenta = excluded flags', 'cyan')]:
    overlay = np.zeros((*mask.shape, 4))
    overlay[find_boundaries(mask, mode='inner')] = [0, 1, 1, 0.8]
    ax.imshow(overlay); ax.set_title(title)
flags = np.zeros((*labels.shape, 4))
flags[find_boundaries(initial_labels, mode='inner') & np.isin(initial_labels, flagged_ids)] = [1, 0, 1, 1]
axes[2].imshow(flags)
plt.show()

## Save the reviewed representative plane

Run this after accepting the settings and inspecting the preceding QC. It saves the already computed arrays without repeating inference. Label IDs and the original pixel grid are preserved. A new output directory prevents overwriting a previous run. Save the original raw projection separately and retain its channel assignments.

In [ ]:
out = OUTPUT / plane
out.mkdir(parents=True, exist_ok=False)
for name, array in [('DAPI_core_enhanced', core), ('unfiltered_labels', initial_labels),
                    ('nuclear_labels', labels), ('tissue_mask', tissue.astype(np.uint8))]:
    path = out / f'{plane}_{name}.tif'
    tifffile.imwrite(path, array, photometric='minisblack',
        metadata={'axes': 'YX', 'unit': 'micron'},
        resolution=(1/cfg['pixel_size_um'], 1/cfg['pixel_size_um']), resolutionunit='NONE')
    assert np.array_equal(tifffile.imread(path), array)
(out / f'{plane}_boundary_qc.json').write_text(json.dumps(boundary_qc, indent=2))
(out / 'segmentation_settings.json').write_text(json.dumps(dict(
    model_class='NuclearSegmentation', model_version='1.1', settings=cfg,
    enhancement=ENHANCEMENT, summary=summary), indent=2))
print('Saved:', out)

## Optional: process the other planes

Use the same loaded model and accepted settings. The convenience function performs enhancement, prediction, filtering and saving, and retains unfiltered predictions and boundary QC records. Review these new outputs before using them for phenotyping. Large enlarged planes can require substantial GPU memory. This batch stage is intentionally separate from the representative-plane review.

In [ ]:
for name in ['top', 'bottom']:
    with tifffile.TiffFile(files[name]) as tf:
        assert tf.series[0].axes == 'CYX'
        image = tf.asarray()
    final, manifest = save_segmentation(
        OUTPUT / name, name, image[DAPI_CHANNEL], app, cfg)
    print(name, manifest['summary'])

## Continue with raw-channel quantification

The raw CYX image and final YX labels share the same pixel grid. Pass both to the measurement module. Each row represents a segmented nucleus, and means come from the unenhanced projected channels. The three planes are independent 2D views; do not sum them as unique cells in the whole tissue. Continue with `Guided_Tutorial.ipynb` for marker gates, anatomical regions and summaries.

In [ ]:
from measure_nuclei import measure_nuclei
from phenotype_analysis import write_csv
channel_names = ['C1 GFP', 'C2 unknown', 'C3 DAPI', 'C4 RFP']
rows = measure_nuclei(raw, labels, channel_names, pixel_size_um=cfg['pixel_size_um'])
for row in rows: row['plane'] = plane
write_csv(out / 'nuclear_channel_means.csv', rows, list(rows[0]) if rows else [])
print('Measured nuclei:', len(rows))